# Portfolio Analysis

**Source of truth:** the curated **beancount ledger** (`ledger/`) — positions are
*derived* from it (cash from the broker snapshot). **Prices:** yfinance, snapshot
fallback · **Classification:** `config/symbol_map.yaml`.

This notebook only **reads** the Parquet the pipeline derives (plus the ledger itself
for point-in-time / counterfactual). Refreshing is done elsewhere: fetch each broker
(`broker-robinhood.ipynb`, `broker-fidelity.ipynb`), then rebuild + verify in
`refresh.ipynb`, then re-run this notebook from the top.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from invest import config, analysis

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

positions = pd.read_parquet(config.POSITIONS_PARQUET)
try:
    history = pd.read_parquet(config.PRICES_PARQUET)
except FileNotFoundError:
    history = pd.DataFrame()

total = positions["market_value"].sum()
print(f"Total portfolio value: ${total:,.2f}")
print(f"Holdings: {len(positions)} across {positions['account_name'].nunique()} accounts "
      f"and {positions['broker'].nunique()} broker(s)")

## Allocation by asset class

In [ ]:
alloc = analysis.allocation_by(positions, "asset_class")
display(alloc)

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(alloc["market_value"], labels=alloc.index,
       autopct=lambda p: f"{p:.1f}%", startangle=90, counterclock=False)
ax.set_title("Allocation by asset class")
plt.show()

## Allocation by account

In [ ]:
acct = analysis.account_summary(positions)
display(acct)

ax = acct["market_value"].plot(kind="barh", figsize=(9, 4), title="Market value by account")
ax.set_xlabel("Market value ($)")
ax.invert_yaxis()
plt.tight_layout(); plt.show()

## Top holdings & concentration

In [ ]:
display(analysis.top_holdings(positions, 10))

c = analysis.concentration(positions)
print(f"HHI (Herfindahl):       {c['hhi']:.3f}   (1.0 = single holding)")
print(f"Effective # of holdings: {c['effective_holdings']:.1f}")
print(f"Top-1 weight:            {c['top1_weight']:.1%}")
print(f"Top-5 weight:            {c['top5_weight']:.1%}")

## Portfolio value over time (all holdings)

Holds **today's** share counts flat across the window and values every holding by
the best source available — actual yfinance history where we have it, a proxy
index for index-tracking funds (shape only), and a flat current value for cash and
proprietary 401k pools with no public quote. The stacked total meets your current
portfolio value at the right edge.

**Risk/composition lens, not performance** — it ignores when you actually bought,
and "flat" holdings show no movement. The coverage line says how much of the path
is real vs. approximated.

In [ ]:
pvh = analysis.portfolio_value_history(positions, history)  # date x asset_class
if pvh.empty:
    print("No price history available — run the pipeline with network enabled.")
else:
    total = pvh.sum(axis=1)

    # Stacked area over every asset class -> the total IS portfolio value.
    ax = pvh.plot.area(figsize=(11, 5), linewidth=0)
    ax.set_title("Portfolio value over time — current holdings, all assets")
    ax.set_ylabel("Value ($)")
    ax.legend(loc="upper left", fontsize=8, ncol=2)
    plt.tight_layout(); plt.show()

    dd = (total / total.cummax() - 1).min()
    print(f"Latest total: ${total.iloc[-1]:,.0f}   |   max drawdown of total: {dd:.1%}")

    # How much of that value path is real vs. approximated.
    tr = analysis.holding_value_treatment(positions, history)
    cov = tr.groupby("treatment")["market_value"].sum()
    pct = (cov / cov.sum() * 100).round(1)
    print("\nPrice-history coverage of current value:")
    for t in ["live", "proxy", "flat"]:
        if t in cov.index:
            print(f"  {t:5}: ${cov[t]:>12,.0f}  ({pct[t]:>5}%)")
    print("  live = actual history · proxy = index stand-in (shape only) · "
          "flat = held at current value (no public history)")

## Benchmark & holding risk summary

In [ ]:
summary = analysis.risk_summary(history)
if summary.empty:
    print("No history loaded.")
else:
    display(summary.sort_values("annual_return", ascending=False)
                   .style.format({"annual_return": "{:.1%}", "annual_vol": "{:.1%}",
                                  "max_drawdown": "{:.1%}", "sharpe_naive": "{:.2f}"}))

## Transactions ledger

The unified `transactions.parquet` — every buy, sell, dividend, interest payment,
transfer and contribution across brokers, normalized to one tidy schema (signed
`amount` = cash flow into the account). `analysis.py` has helpers over it:
`income_by_period`, `cash_flows`, `fees_paid`,
`dividend_calendar`, `contributions_summary` — slice it however you like.

In [ ]:
# The ledger flattened to events (derived from the beancount source of truth).
from invest import ledger
entries, _, _ = ledger.load()

tx = pd.read_parquet(config.TRANSACTIONS_PARQUET)
print(f"{len(tx):,} events · {tx['date'].min():%Y-%m-%d} → {tx['date'].max():%Y-%m-%d} · "
      f"brokers: {', '.join(sorted(tx['broker'].dropna().unique()))}")
display(tx["type"].value_counts().to_frame("events").T)

# Income (dividends + interest) by year
income = analysis.income_by_period(tx, freq="Y")
if not income.empty:
    display(income.style.format("${:,.0f}"))
    cols = [c for c in ("dividend", "interest") if c in income.columns]
    ax = income[cols].plot(kind="bar", stacked=True, figsize=(10, 4),
                           title="Income by year (dividends + interest)")
    ax.set_ylabel("$"); ax.set_xlabel(""); plt.tight_layout(); plt.show()

# Realized gains — straight from beancount's lot tracking (full cost basis, including
# the curated opening lots), then contributions by account.
rg = ledger.realized_gains_dataframe(entries)
print(f"Realized P/L: ${rg['realized'].sum():,.0f} across {len(rg)} sales")
display(rg.groupby("symbol")["realized"].sum().sort_values()
          .to_frame("realized").style.format("${:,.0f}"))
display(analysis.contributions_summary(tx))

## Point-in-time & counterfactual

Because positions are a **fold over the dated ledger**, we can realize holdings *as of
any past date*, and *edit the event stream* (drop/alter trades) and re-derive — the
questions a snapshot can't answer. `ledger.positions_dataframe(entries, date=…)` and
`ledger.filter_entries(entries, predicate)` are the primitives.

In [ ]:
# Point-in-time: realize share holdings as of past dates (a fold over the dated ledger).
dates = ["2023-06-05", "2024-06-05", "2025-06-05", None]
frames = {}
for d in dates:
    p = ledger.positions_dataframe(entries, date=d)
    frames[d or "today"] = p.loc[~p["is_cash"]].groupby("symbol")["quantity"].sum()
pit = pd.DataFrame(frames).fillna(0).round(2)
display(pit.loc[pit.abs().sum(axis=1) > 0].sort_values("today", ascending=False).head(12))

# Counterfactual: what if you'd never sold NVDA? Drop NVDA sells and re-derive.
cf = ledger.filter_entries(entries, lambda t: not (
    t.meta.get("txn_type") == "sell" and t.meta.get("symbol") == "NVDA"))
actual = ledger.positions_dataframe(entries).query("symbol == 'NVDA'")["quantity"].sum()
nosell = ledger.positions_dataframe(cf).query("symbol == 'NVDA'")["quantity"].sum()
print(f"NVDA shares — actual: {actual:.2f} · if never sold: {nosell:.2f}")

---
*yfinance is unofficial — treat prices as research-grade. Positions are derived from
the curated beancount ledger (`ledger/`); accuracy depends on opening-lot/split fixups
being curated — the pipeline prints which symbols still need them.*